# 歌词分词，词性标注

In [1]:
import json
import pandas as pd


# import thulac


from collections import Counter
from openai import OpenAI

In [2]:
# 可以选择是否加载
# jieba.load_userdict('data/mayday_dict_simple.txt')

In [3]:
import sys
sys.path.append('..')

# 分词，词频与词性分析

In [4]:
word_to_fix = {
    '阮': 'r',
    '袂': 'v',
    '春娇': 'n',
    '学会': 'v'
}

In [5]:
# def process_lyrics_with_jieba(text):
#     # 1. 词性标注与分词
#     # jieba.posseg 会同时返回词和词性
#     words_with_pos = pseg.cut(text)

    
#     # 2. 过滤无意义字符（标点、空格、单字符停用词）
#     filtered_data = []
#     for word, pos in words_with_pos:
#         # 排除标点符号（x表示标点）及空白字符
#         if pos != 'x' and len(word.strip()) > 0:
#             if word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 3. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 4. 汇总信息 (词, 词性, 频数)
#     # 我们以词为 Key，存储词性
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     # 排序：按词频从高到低
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count # 词频
#         })
    
#     return sorted_results

In [6]:
import re
from hanlp_restful import HanLPClient
HanLP = HanLPClient('https://www.hanlp.com/hanlp/v21/redirect', auth="699691e7eaf61a3aca90d7b8", language='zh')

def is_chinese_word(word):
    """
    判断是否为纯中文词
    """
    return 1 if re.fullmatch(r'[\u4e00-\u9fff]+', word) else 0


def process_lyrics_with_hanlp_multi_pos(text, word_to_fix=None):
    if not text:
        return []
    
    # 调用 HanLP
    result = HanLP.parse(text, tasks='pos/pku')
    
    sentences = result['tok/fine']
    pos_sentences = result['pos/pku']
    
    # 统计 (word, pos) -> freq
    word_pos_counter = Counter()
    
    for words, pos_tags in zip(sentences, pos_sentences):
        for word, tag in zip(words, pos_tags):
            
            word = word.strip()
            
            # 过滤标点
            if tag == 'w' or not word:
                continue
            
            # 词性修正
            if word_to_fix and word in word_to_fix:
                tag = word_to_fix[word]
            
            word_pos_counter[(word, tag)] += 1
    
    # 构建结果列表
    results = []
    for (word, pos), freq in word_pos_counter.items():
        results.append({
            "word": word,
            "pos": pos,
            "freq": freq,
            "is_chinese": is_chinese_word(word)
        })
    
    # 按词频排序
    results.sort(key=lambda x: x["freq"], reverse=True)
    
    return results


In [7]:
# thu = thulac.thulac(seg_only=False, filt=True) 

# def process_lyrics_with_thulac(text, word_to_fix=None):
#     if not text:
#         return []
    
#     # 2. 执行分词与词性标注
#     # 返回格式为 [[word, pos], [word, pos], ...]
#     words_with_pos = thu.cut(text)
    
#     # 3. 过滤无意义字符与词性修正
#     # thulac 的标点词性通常是 'w'
#     filtered_data = []
#     for word, pos in words_with_pos:
#         word = word.strip()
#         # 排除标点符号、空白字符
#         if pos != 'w' and len(word) > 0:
#             # 逻辑修正：word_to_fix 通常是修正词性
#             if word_to_fix and word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 4. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 5. 汇总信息
#     # 建立 word -> pos 映射
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count
#         })
    
#     return sorted_results

In [8]:
def lyric_words_process(path_prefix, word_to_fix=None):
    lyric_file_path = path_prefix + 'cleared_lyric_data.json'
    # 读取歌词文件
    with open(lyric_file_path, 'r') as f:
        lyric_data = json.load(f)
    lyric_words_dict = {}
    for i in lyric_data:
        if i:
            # lyric_words_dict[i['song_id']] = process_lyrics_with_jieba(
            #     i['lyrics_text'])
            # lyric_words_dict[i['song_id']] = process_lyrics_with_thulac(
            #     i['lyrics_text'], word_to_fix=word_to_fix)
            print(i['song_name'])
            lyric_words_dict[i['song_id']] = process_lyrics_with_hanlp_multi_pos(
                i['lyrics_text'], word_to_fix=word_to_fix)
    rows = []
    for song_id, word_list in lyric_words_dict.items():
        for item in word_list:
            # 创建新字典，保留原始数据并加入歌曲ID列
            new_row = {
                'song_id': song_id,
                'word': item['word'],
                'pos': item['pos'],
                'freq': item['freq']
            }
            rows.append(new_row)

    # 3. 转换为 DataFrame
    df_word = pd.DataFrame(rows)
    return df_word

In [9]:
def words_data_merge(df_word, df_songs):
    # 合并
    # 1. 确保 df_word 的 song_id 是字符串
    df_word = df_word.copy()
    df_songs = df_songs.copy()
    df_word['song_id'] = df_word['song_id'].astype(str)
    df_word['is_chinese'] = df_word['word'].apply(is_chinese_word)

    # 2. 确保 df_unique 的 song_id 是字符串（并去掉可能存在的空格）
    df_songs['song_id'] = df_songs['song_id'].astype(str).str.strip()

    # 3. 执行合并
    df_merged = df_word.merge(df_songs, on='song_id', how='left')

    # 4. 删除空值
    # df_merged = df_merged.dropna()

    return df_merged

# main

In [96]:
singer_list = [
        'mayday', 'jaychou', 'liyuchun', 'chenyixun', 'renxianqi', 'linjunjie',
        'sunyanzi', 'remen', 'fangwenshan', 'chenxinhong', 'caiyilin', 'wubai', 'zhoushen', 'zhoushen_pure'
    ]
file_path_prefix = f"data/{singer_list[-1]}/"
# file_path_prefix = f"data/renxianqi/"

In [97]:
# 歌曲数据
df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,ost_name,song_name_unique,song_name_pure,publish_date,publish_year
0,214188340,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,003fA5G40k6hKc,云裳羽衣曲,4096700,001uf8A626x0yY,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018
1,468749280,002QvWwX1BG9mr,小美满,《热辣滚烫》电影热辣陪伴曲,周深,199509,003fA5G40k6hKc,小美满,46467243,0002ATV00WcBgd,214,1707184800,热辣滚烫,小美满,小美满,2024-02-06,2024
2,106484214,004OQ5Mt0EmEzv,大鱼,《大鱼海棠》动画电影印象曲,周深,199509,003fA5G40k6hKc,大鱼,1387962,004Y7V4s3ug4cC,313,1463673600,大鱼海棠,大鱼,大鱼,2016-05-20,2016
3,639981206,002LbZMr3NfNPI,选择,《惊蛰无声》电影主题曲,周深,199509,003fA5G40k6hKc,惊蛰无声 影视原声带,84974077,002jh3xv1whZS8,203,1771293600,惊蛰无声,选择,选择,2026-02-17,2026
4,297971384,002Id4Xd0F98go,若梦,《与君歌》电视剧主题曲,周深,199509,003fA5G40k6hKc,若梦,17590611,0025JWkx1h0vuU,244,1613268000,与君歌,若梦,若梦,2021-02-14,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,360994402,000WYGmR2DxPPg,万里挑一,《关于唐医生的一切》电视剧主题曲,周深,199509,003fA5G40k6hKc,关于唐医生的一切 电视原声带,28839736,001Zftkf4Yh2W2,222,1656172800,关于唐医生的一切,万里挑一,万里挑一,2022-06-26,2022
126,451400575,0016s8pN4cpBxw,I'm A Star,《星愿》电影许愿星之歌,周深,199509,003fA5G40k6hKc,I'm A Star,44068462,0007r8Wy0nIyAV,187,1700409600,星愿,I'mAStar,i'mastar,2023-11-20,2023
127,356161397,002cue0j1dvmxt,I See Us,《欢迎光临》电视剧主题曲,周深,199509,003fA5G40k6hKc,欢迎光临 电视剧原声带,28134589,003EADJV2YYcIe,256,1652839200,欢迎光临,ISeeUs,iseeus,2022-05-18,2022
128,257720758,000Ji8co0G4eU3,启示,《地下城与勇士·逆转之轮》推广曲,周深,199509,003fA5G40k6hKc,DNF官方动画第二季推广曲,11296569,004A5e9i2qtWrZ,280,1586707200,地下城与勇士·逆转之轮,启示,启示,2020-04-13,2020


In [ ]:
# 五月天需要使用word_to_fix
# if file_path_prefix == "data/mayday/":
#     df_word = lyric_words_process(file_path_prefix, word_to_fix)
# else:
#     df_word = lyric_words_process(file_path_prefix, word_to_fix=None)

In [85]:
# 词性解析
# hanlp暂时不需要word_to_fix
df_word = lyric_words_process(file_path_prefix, word_to_fix=None)
df_word.to_csv(file_path_prefix + "raw_words_data.csv", index=False)
df_word

云裳羽衣曲
小美满
大鱼
选择
若梦
璀璨冒险人
万里
光亮
我的对
谜宫
借过一下
时间啊
和光同尘
Crush
遥遥
有我
人是_
梦见你
我以渺小爱你
心海里的光
等光来
消散人潮
触不可及
浮游
Rubia
冰凌花
铃芽之旅
芽
生活总该迎着光亮
浣花落
My Only
要一起
风吹过的晨曦
归来
随风
借梦
余情
若以尘埃
相拥不放
河
门
向光而行
焰火
愿
后来没有你
岁月尘埃
茧
光字片
独白
愿得一心人
不再流浪
来不及勇敢
悬崖之上
曼陀
海藏
玦恋
缘起
直破穹苍
旅途
与卿
雪花落下
画绢
问花
请笃信一个梦
行舟问柳
明月传说
【薛洋】 荒城渡
以无旁骛之吻
拙慕
明明
如你随行
一生一瞬
桂花谣
你的样子
追赶春天的人
相守
若仙
在意
许卿安
念归去
痕迹
曾经沧海
卡布叻船长
繁花依旧
水形物语
江湖觅知音
如果你爱我
时间之海
她说，老啦
迷途
人间星河
星鱼
归处
小舍得
说声你好
明暗之间
也很值得
鲛人之歌
心事
逐月
此生惟你
威凤吟
蜕
永恒孤独
最好的礼物
夏日友晴天
过客
彼岸花
非你所想
东游
浓情淡如你
一晌
她
战·永不言败
风起流年
天地为念
跳舞的月光
回到你身边
情意结
记得
Endless Sailing
请带着浪漫远航
繁星璀璨的天空
卿卿
不说话
万里挑一
I'm A Star
I See Us
启示
照耀星河


,song_id,word,pos,freq
0,214188340,的,u,12
1,214188340,我,r,10
2,214188340,你,r,9
3,214188340,故事,n,6
4,214188340,鲜艳,a,6
...,...,...,...,...
12483,353730842,留给,v,2
12484,353730842,岁月,n,2
12485,353730842,一抹,m,2
12486,353730842,温柔,a,2


In [98]:
# 重新读取
df_word_read = pd.read_csv(file_path_prefix + "raw_words_data.csv")

In [99]:
df_merged = words_data_merge(df_word_read, df_songs)
df_merged = df_merged.dropna(subset='song_name')
df_merged

,song_id,word,pos,freq,is_chinese,song_mid,song_name,song_subname,artist_name,artist_id,...,album_name,album_id,album_mid,duration,publish_time,ost_name,song_name_unique,song_name_pure,publish_date,publish_year
0,214188340,的,u,12,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,云裳羽衣曲,4096700,001uf8A626x0yY,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018
1,214188340,我,r,10,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,云裳羽衣曲,4096700,001uf8A626x0yY,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018
2,214188340,你,r,9,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,云裳羽衣曲,4096700,001uf8A626x0yY,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018
3,214188340,故事,n,6,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,云裳羽衣曲,4096700,001uf8A626x0yY,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018
4,214188340,鲜艳,a,6,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,云裳羽衣曲,4096700,001uf8A626x0yY,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12483,353730842,留给,v,2,1,0019oYqI1AgmOA,照耀星河,《良辰好景知几何》电视剧主题曲,周深,199509,...,良辰好景知几何 电视剧原声带,27466271,0023aWMA30ro5t,266,1652371200,良辰好景知几何,照耀星河,照耀星河,2022-05-13,2022
12484,353730842,岁月,n,2,1,0019oYqI1AgmOA,照耀星河,《良辰好景知几何》电视剧主题曲,周深,199509,...,良辰好景知几何 电视剧原声带,27466271,0023aWMA30ro5t,266,1652371200,良辰好景知几何,照耀星河,照耀星河,2022-05-13,2022
12485,353730842,一抹,m,2,1,0019oYqI1AgmOA,照耀星河,《良辰好景知几何》电视剧主题曲,周深,199509,...,良辰好景知几何 电视剧原声带,27466271,0023aWMA30ro5t,266,1652371200,良辰好景知几何,照耀星河,照耀星河,2022-05-13,2022
12486,353730842,温柔,a,2,1,0019oYqI1AgmOA,照耀星河,《良辰好景知几何》电视剧主题曲,周深,199509,...,良辰好景知几何 电视剧原声带,27466271,0023aWMA30ro5t,266,1652371200,良辰好景知几何,照耀星河,照耀星河,2022-05-13,2022


In [100]:
# 过滤中文词
df_merged_chn = df_merged[df_merged['is_chinese'] == 1]
df_merged_chn

,song_id,word,pos,freq,is_chinese,song_mid,song_name,song_subname,artist_name,artist_id,...,album_name,album_id,album_mid,duration,publish_time,ost_name,song_name_unique,song_name_pure,publish_date,publish_year
0,214188340,的,u,12,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,云裳羽衣曲,4096700,001uf8A626x0yY,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018
1,214188340,我,r,10,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,云裳羽衣曲,4096700,001uf8A626x0yY,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018
2,214188340,你,r,9,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,云裳羽衣曲,4096700,001uf8A626x0yY,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018
3,214188340,故事,n,6,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,云裳羽衣曲,4096700,001uf8A626x0yY,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018
4,214188340,鲜艳,a,6,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,云裳羽衣曲,4096700,001uf8A626x0yY,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12483,353730842,留给,v,2,1,0019oYqI1AgmOA,照耀星河,《良辰好景知几何》电视剧主题曲,周深,199509,...,良辰好景知几何 电视剧原声带,27466271,0023aWMA30ro5t,266,1652371200,良辰好景知几何,照耀星河,照耀星河,2022-05-13,2022
12484,353730842,岁月,n,2,1,0019oYqI1AgmOA,照耀星河,《良辰好景知几何》电视剧主题曲,周深,199509,...,良辰好景知几何 电视剧原声带,27466271,0023aWMA30ro5t,266,1652371200,良辰好景知几何,照耀星河,照耀星河,2022-05-13,2022
12485,353730842,一抹,m,2,1,0019oYqI1AgmOA,照耀星河,《良辰好景知几何》电视剧主题曲,周深,199509,...,良辰好景知几何 电视剧原声带,27466271,0023aWMA30ro5t,266,1652371200,良辰好景知几何,照耀星河,照耀星河,2022-05-13,2022
12486,353730842,温柔,a,2,1,0019oYqI1AgmOA,照耀星河,《良辰好景知几何》电视剧主题曲,周深,199509,...,良辰好景知几何 电视剧原声带,27466271,0023aWMA30ro5t,266,1652371200,良辰好景知几何,照耀星河,照耀星河,2022-05-13,2022


In [101]:
# 查看歌曲数
df_merged_chn['song_name_pure'].nunique()

126

In [102]:
# 虚拟专辑数据
df_songs_part = df_merged_chn[[
    'song_name_pure'
]].drop_duplicates(keep='first').reset_index(drop=True)
# 只保留120个
df_songs_part = df_songs_part.head(120)
df_songs_part['album_name'] = "PART " + (df_songs_part.index // 10 +
                                         1).astype(str)
df_songs_part['album_order'] = df_songs_part.index // 10
df_songs_part

,song_name_pure,album_name,album_order
0,云裳羽衣曲,PART 1,0
1,小美满,PART 1,0
2,大鱼,PART 1,0
3,选择,PART 1,0
4,若梦,PART 1,0
...,...,...,...
115,回到你身边,PART 12,11
116,情意结,PART 12,11
117,记得,PART 12,11
118,请带着浪漫远航,PART 12,11


In [103]:
# 虚拟专辑数据，index//12+1作为虚拟专辑
df_merged_chn = df_merged_chn.copy()
df_merged_chn['album_name_raw'] = df_merged_chn['album_name']
df_merged_chn = df_merged_chn.drop(columns=['album_name'])
df_merged_chn = df_merged_chn.merge(df_songs_part, on='song_name_pure', how='left')
# 删除album_order为空的数据
df_merged_chn = df_merged_chn.dropna(subset=['album_order'], axis=0)
df_merged_chn

,song_id,word,pos,freq,is_chinese,song_mid,song_name,song_subname,artist_name,artist_id,...,duration,publish_time,ost_name,song_name_unique,song_name_pure,publish_date,publish_year,album_name_raw,album_name,album_order
0,214188340,的,u,12,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018,云裳羽衣曲,PART 1,0.0
1,214188340,我,r,10,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018,云裳羽衣曲,PART 1,0.0
2,214188340,你,r,9,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018,云裳羽衣曲,PART 1,0.0
3,214188340,故事,n,6,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018,云裳羽衣曲,PART 1,0.0
4,214188340,鲜艳,a,6,1,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,...,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018,云裳羽衣曲,PART 1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11406,313505407,点燃,v,1,1,003dCRtx3lBJ81,繁星璀璨的天空,《光荣与梦想》电视剧致敬曲,周深,199509,...,241,1623204000,光荣与梦想,繁星璀璨的天空,繁星璀璨的天空,2021-06-09,2021,光荣与梦想 电视剧原声大碟,PART 12,11.0
11407,313505407,生命,n,1,1,003dCRtx3lBJ81,繁星璀璨的天空,《光荣与梦想》电视剧致敬曲,周深,199509,...,241,1623204000,光荣与梦想,繁星璀璨的天空,繁星璀璨的天空,2021-06-09,2021,光荣与梦想 电视剧原声大碟,PART 12,11.0
11408,313505407,引爆,v,1,1,003dCRtx3lBJ81,繁星璀璨的天空,《光荣与梦想》电视剧致敬曲,周深,199509,...,241,1623204000,光荣与梦想,繁星璀璨的天空,繁星璀璨的天空,2021-06-09,2021,光荣与梦想 电视剧原声大碟,PART 12,11.0
11409,313505407,夜,Tg,1,1,003dCRtx3lBJ81,繁星璀璨的天空,《光荣与梦想》电视剧致敬曲,周深,199509,...,241,1623204000,光荣与梦想,繁星璀璨的天空,繁星璀璨的天空,2021-06-09,2021,光荣与梦想 电视剧原声大碟,PART 12,11.0


In [104]:
df_merged_chn.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)

In [105]:
# 数据查验
songs_n = df_merged_chn[df_merged_chn['pos'] == 'n']['song_name_pure'].unique().tolist()
songs_all = df_merged_chn['song_name_pure'].unique().tolist()
for i in songs_all:
    if i not in songs_n:
        print(i)

# 歌曲数据更新

In [106]:
df_songs_final = df_merged_chn.drop(columns=['word', 'pos', 'freq', 'is_chinese']).drop_duplicates().reset_index(drop=True)

df_songs_final

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_id,album_mid,duration,publish_time,ost_name,song_name_unique,song_name_pure,publish_date,publish_year,album_name_raw,album_name,album_order
0,214188340,000wXRGx2u2NJ8,云裳羽衣曲,《云裳羽衣》手游首发主题曲,周深,199509,003fA5G40k6hKc,4096700,001uf8A626x0yY,296,1530028800,云裳羽衣,云裳羽衣曲,云裳羽衣曲,2018-06-27,2018,云裳羽衣曲,PART 1,0.0
1,468749280,002QvWwX1BG9mr,小美满,《热辣滚烫》电影热辣陪伴曲,周深,199509,003fA5G40k6hKc,46467243,0002ATV00WcBgd,214,1707184800,热辣滚烫,小美满,小美满,2024-02-06,2024,小美满,PART 1,0.0
2,106484214,004OQ5Mt0EmEzv,大鱼,《大鱼海棠》动画电影印象曲,周深,199509,003fA5G40k6hKc,1387962,004Y7V4s3ug4cC,313,1463673600,大鱼海棠,大鱼,大鱼,2016-05-20,2016,大鱼,PART 1,0.0
3,639981206,002LbZMr3NfNPI,选择,《惊蛰无声》电影主题曲,周深,199509,003fA5G40k6hKc,84974077,002jh3xv1whZS8,203,1771293600,惊蛰无声,选择,选择,2026-02-17,2026,惊蛰无声 影视原声带,PART 1,0.0
4,297971384,002Id4Xd0F98go,若梦,《与君歌》电视剧主题曲,周深,199509,003fA5G40k6hKc,17590611,0025JWkx1h0vuU,244,1613268000,与君歌,若梦,若梦,2021-02-14,2021,若梦,PART 1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,339869799,000l48yc20YX8i,回到你身边,《奇迹·笨小孩》电影新愿曲,周深,199509,003fA5G40k6hKc,28634784,001ipqSV1hxylV,244,1642730400,奇迹·笨小孩,回到你身边,回到你身边,2022-01-21,2022,奇迹·笨小孩 电影原声大碟,PART 12,11.0
116,237538300,003danDK4Zcqq9,情意结,《诛仙I》电影片尾曲,周深,199509,003fA5G40k6hKc,7981965,004OLQfC3d8bqt,270,1567699200,诛仙I,情意结,情意结,2019-09-06,2019,诛仙I 电影原声带,PART 12,11.0
117,481321546,000dluoh14RTis,记得,《承欢记》电视剧心动主题曲,周深,199509,003fA5G40k6hKc,48830692,001Ite173BdBCZ,196,1713319200,承欢记,记得,记得,2024-04-17,2024,承欢记 电视剧影视原声带,PART 12,11.0
118,362521047,003yD9J81jUPPu,请带着浪漫远航,《冲出地球》电影主题曲,周深,199509,003fA5G40k6hKc,29278578,0049rHX01ixcc9,233,1657245600,冲出地球,请带着浪漫远航,请带着浪漫远航,2022-07-08,2022,冲出地球 电影原声音乐大碟,PART 12,11.0


In [107]:
df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

# 测试

In [ ]:
160*4